In [1]:
from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Dict, TextIO, List


def split_jsonl_to_csv_by_arxiv_category(
    in_path: str | Path,
    out_dir: str | Path,
    *,
    nest_by_top_level: bool = True,
    unknown_bucket: str = "_unknown",
) -> None:
    """
    Splits JSONL into one CSV per arxiv_primary_category.
    CSV headers are the union of keys seen per category (computed on the fly per file).
    """
    in_path = Path(in_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # For each output file: keep (file_handle, csv_writer, fieldnames)
    handles: Dict[Path, TextIO] = {}
    writers: Dict[Path, csv.DictWriter] = {}
    fieldnames_map: Dict[Path, List[str]] = {}

    def out_path_for(cat: str) -> Path:
        cat = (cat or "").strip() or unknown_bucket
        top = cat.split(".", 1)[0]
        if nest_by_top_level:
            subdir = out_dir / top
            subdir.mkdir(parents=True, exist_ok=True)
            return subdir / f"{cat}.csv"
        return out_dir / f"{cat}.csv"

    def ensure_writer(path: Path, obj: dict) -> csv.DictWriter:
        if path not in handles:
            handles[path] = path.open("w", encoding="utf-8", newline="")
            fieldnames_map[path] = list(obj.keys())
            w = csv.DictWriter(handles[path], fieldnames=fieldnames_map[path], extrasaction="ignore")
            w.writeheader()
            writers[path] = w
        else:
            # If new keys appear later, we *cannot* easily rewrite header without re-writing file.
            # If you expect variable keys, prefer JSONL output.
            pass
        return writers[path]

    try:
        with in_path.open("r", encoding="utf-8") as f:
            for lineno, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError as e:
                    raise ValueError(f"Bad JSON on line {lineno}: {e}") from e

                cat = (obj.get("arxiv_primary_category") or "").strip() or unknown_bucket
                path = out_path_for(cat)
                w = ensure_writer(path, obj)
                w.writerow(obj)

    finally:
        for fp in handles.values():
            fp.close()


# Example usage:
split_jsonl_to_csv_by_arxiv_category("equations.jsonl", "split_csv", nest_by_top_level=True)

FileNotFoundError: [Errno 2] No such file or directory: 'equations.jsonl'